# 🚀 Модуль 3. Урок 1. Python в эпоху AI: отказ от зубрежки синтаксиса в пользу логических структур

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_03_python/lesson_01_python_no_syntax_memorization.ipynb)

## 📌 Бизнес-контекст и методология
В 2027 году аналитику больше не нужно держать в памяти сигнатуры всех методов Pandas. LLM мгновенно генерируют код по описанию задачи.

**Что критически важно понимать человеку:**
1. **Векторизация vs циклы**: операции в Pandas и NumPy выполняются на уровне C-массивов (`O(1)` в интерпретаторе).
2. **Мутабельность и `.copy()`**: избегание предупреждения `SettingWithCopyWarning`.
3. **Метод декомпозиции**: последовательность трансформаций (Extract → Filter → Transform → Aggregate).

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import time
import numpy as np
import pandas as pd

n_elements = 1_000_000
data_arr = np.random.uniform(10, 1000, n_elements)

start_time = time.time()
res_loop = [x * 1.20 for x in data_arr]
loop_duration = time.time() - start_time

start_time = time.time()
res_vector = data_arr * 1.20
vec_duration = time.time() - start_time

print(f"⏱️ Время цикла for:       {loop_duration:.4f} сек")
print(f"⚡ Время векторизации:    {vec_duration:.4f} сек")
print(f"🚀 Векторизация быстрее в {loop_duration / vec_duration:.1f} раз!")


⏱️ Время цикла for:       0.1294 сек
⚡ Время векторизации:    0.0019 сек
🚀 Векторизация быстрее в 69.4 раз!


### 🎯 Практическое задание 1: Векторный расчет НДС и маржи
Создайте датафрейм из 500 000 чеков со случайными суммами от 500 до 15 000 руб.
Рассчитайте векторным способом выделенный НДС 20% и сумму без НДС.

In [3]:
### ВАШ КОД ЗДЕСЬ
import time
import pandas as pd
import numpy as np

np.random.seed(42)
df_sales = pd.DataFrame({
    'gross_amount': np.random.uniform(500, 15000, 500_000)
})

t0 = time.time()
df_sales['vat_amount'] = df_sales['gross_amount'] * 20 / 120
df_sales['net_amount'] = df_sales['gross_amount'] - df_sales['vat_amount']
calc_time = time.time() - t0

print(f"Расчет выполнен за: {calc_time:.4f} сек.")
assert calc_time < 0.25, "Расчет должен быть векторизован"
assert np.isclose(df_sales['net_amount'].sum() + df_sales['vat_amount'].sum(), df_sales['gross_amount'].sum()), "Баланс должен сходиться"
print("✅ Тест успешно пройден: векторизация освоена!")


Расчет выполнен за: 0.0054 сек.
✅ Тест успешно пройден: векторизация освоена!


### 💡 Промпт для Cursor / Copilot
> *«Напиши векторизованный расчет скидок по условию с использованием `np.select`, избегая `df.apply` и циклов `iterrows()`.»*